# TimesFM — Walmart Store Sales Forecasting

## 1. Dependency 
Kaggle-ის `transformers`/`safetensors` ხშირად იწვევს (`safe_open backend` შეცდომას). ეს cell-ი ასწორებს ამ პრობლემას. გაშვების შემდეგ **restart** (Run → Restart Session),

In [ ]:
%pip install -q --force-reinstall "safetensors>=0.5.3"
import safetensors
print("safetensors:", safetensors.__version__)
print(">>> RESTART KERNEL NOW (Run -> Restart Session), then run from the next section <<<")

## MLflow / DagsHub ინიციალიზაცია და GitHub-დან preprocessing-ის იმპორტი


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip


In [2]:
import numpy as np
import pandas as pd
import torch
torch.set_float32_matmul_precision("high")
print("cuda:", torch.cuda.is_available())

cuda: True


In [ ]:
import transformers, safetensors
print("transformers:", transformers.__version__, "| safetensors:", safetensors.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

transformers: 4.44.2 | safetensors: 0.8.0


In [4]:
!wget -q -O preprocessing.py "https://raw.githubusercontent.com/IrakliZerekidze/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting/main/preprocessing.py"
from preprocessing import run_pipeline, weighted_mae

In [10]:
%pip install -q dagshub mlflow
import dagshub
dagshub.init(repo_owner='izere23',
             repo_name='ML-Final-Walmart-Recruiting-Store-Sales-Forecasting',
             mlflow=True)
import mlflow
mlflow.set_experiment("TimesFM_Training")

Note: you may need to restart the kernel to use updated packages.


Initialized MLflow to track repo "izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting"

Repository izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting initialized!

<Experiment: artifact_location='mlflow-artifacts:/13cda5bbe997415cb20ef7866b21e8f3', creation_time=1783860972220, effective_trace_archival_retention=None, experiment_id='8', last_update_time=1783860972220, lifecycle_stage='active', name='TimesFM_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

## მონაცემების ჩატვირთვა და train/validation split

In [6]:
train_part, valid_part, train_full, test_full = run_pipeline(
    data_dir="/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting",
    out_dir="/kaggle/working/processed",
)
for df in (train_part, valid_part):
    if "IsHoliday" not in df.columns and "IsHoliday_x" in df.columns:
        df.rename(columns={"IsHoliday_x": "IsHoliday"}, inplace=True)
        df.drop(columns=["IsHoliday_y"], inplace=True)

TARGET = "Weekly_Sales"
train = train_part.dropna(subset=[TARGET]).copy()
valid = valid_part.dropna(subset=[TARGET]).copy()
train["unique_id"] = train["Store"].astype(str) + "_" + train["Dept"].astype(str)
valid["unique_id"] = valid["Store"].astype(str) + "_" + valid["Dept"].astype(str)

long_train = (train.rename(columns={"Date": "ds", TARGET: "y"})[["unique_id", "ds", "y"]]
                   .sort_values(["unique_id", "ds"]).reset_index(drop=True))
print("train series:", long_train["unique_id"].nunique(), "| val rows:", len(valid))

Expected rows if no gaps: 428409
Actual rows: 380107
Missing (Store,Dept,Date) combos filled in: 48302
Expected rows if no gaps: 476333
Actual rows: 421570
Missing (Store,Dept,Date) combos filled in: 54763
Saved parquet files to /kaggle/working/processed
train series: 3321 | val rows: 41463


## TimesFM 2.0-ის ჩატვირთვა


In [ ]:
HORIZON = 24
PREDICT = None

try:
    import timesfm
    attrs = [a for a in dir(timesfm) if not a.startswith("_")]
    print("timesfm attrs:", attrs)

    if hasattr(timesfm, "TimesFM_2p5_200M_torch"):
        _m = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
        _m.compile(timesfm.ForecastConfig(max_context=1024, max_horizon=max(HORIZON, 64),
                                          normalize_inputs=True, infer_is_positive=True))
        def PREDICT(arrays):
            pf, _ = _m.forecast(horizon=HORIZON, inputs=arrays)
            return np.asarray(pf)[:, :HORIZON]
        print("loaded via timesfm 2.5 API")

    elif hasattr(timesfm, "TimesFm"):
        _m = timesfm.TimesFm(
            hparams=timesfm.TimesFmHparams(
                backend=("gpu" if torch.cuda.is_available() else "cpu"),
                per_core_batch_size=32, horizon_len=HORIZON,
                input_patch_len=32, output_patch_len=128,
                num_layers=50, model_dims=1280, use_positional_embedding=False),
            checkpoint=timesfm.TimesFmCheckpoint(
                huggingface_repo_id="google/timesfm-2.0-500m-pytorch"))
        def PREDICT(arrays):
            pf, _ = _m.forecast(inputs=arrays, freq=[1]*len(arrays))
            return np.asarray(pf)[:, :HORIZON]
        print("loaded via timesfm 2.0 API")
except Exception as e:
    print("native timesfm loader failed:", repr(e))

if PREDICT is None:
    from transformers import TimesFmModelForPrediction
    _hf = TimesFmModelForPrediction.from_pretrained(
        "google/timesfm-2.0-500m-pytorch", attn_implementation="sdpa", device_map="auto")
    _hf.eval()
    _dev = next(_hf.parameters()).device
    @torch.no_grad()
    def PREDICT(arrays):
        inp = [torch.tensor(a, device=_dev) for a in arrays]
        freq = torch.zeros(len(arrays), dtype=torch.long, device=_dev)
        out = _hf(past_values=inp, freq=freq, return_dict=True)
        return out.mean_predictions.float().cpu().numpy()[:, :HORIZON]
    print("loaded via HuggingFace Transformers API")

assert PREDICT is not None, "no TimesFM loader worked"

_test = PREDICT([np.sin(np.linspace(0, 20, 120)).astype(np.float32)])
print("smoke test output shape:", np.asarray(_test).shape)

timesfm attrs: ['ForecastConfig', 'TimesFM_2p5_200M_torch', 'configs', 'timesfm_2p5', 'timesfm_2p5_torch', 'torch']


config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/925M [00:00<?, ?B/s]

loaded via timesfm 2.5 API
smoke test output shape: (1, 24)


## Experiments - MLFlow Log

In [ ]:
CUTOFF = long_train["ds"].max()
G_MEAN = long_train["y"].mean()
SD_MEAN = long_train.groupby("unique_id")["y"].mean().to_dict()

def run_timesfm(run_name, context_len=None, log_space=False, horizon=HORIZON):
    ctx = {}
    for u, g in long_train.groupby("unique_id"):
        arr = g["y"].to_numpy(dtype=np.float32)
        if context_len:
            arr = arr[-context_len:]
        if log_space:
            arr = np.log1p(np.clip(arr, 0, None))
        ctx[u] = arr
    uids = list(ctx)

    preds_by_uid, B = {}, 256
    for i in range(0, len(uids), B):
        chunk = uids[i:i+B]
        fc = np.asarray(PREDICT([ctx[u] for u in chunk]))
        if log_space:
            fc = np.expm1(fc)
        for u, row in zip(chunk, fc):
            preds_by_uid[u] = row

    def lookup(u, k):
        row = preds_by_uid.get(u)
        return SD_MEAN.get(u, G_MEAN) if (row is None or k < 1 or k > horizon) else row[k-1]

    va = valid[["unique_id", "Store", "Dept", "Date", "IsHoliday", TARGET]].copy()
    va["k"] = ((va["Date"] - CUTOFF).dt.days // 7).astype(int)
    pred = np.clip([lookup(u, k) for u, k in zip(va["unique_id"], va["k"])], 0, None)

    matched = int(sum((preds_by_uid.get(u) is not None) and (1 <= k <= horizon)
                      for u, k in zip(va["unique_id"], va["k"])))
    n_fallback = len(va) - matched

    val_wmae = weighted_mae(va[TARGET].values, pred, va["IsHoliday"].values)
    val_mae  = float(np.mean(np.abs(va[TARGET].values - pred)))
    val_rmse = float(np.sqrt(np.mean((va[TARGET].values - pred) ** 2)))
    hol = va["IsHoliday"].values.astype(bool)
    hol_wmae = weighted_mae(va[TARGET].values[hol], pred[hol], va["IsHoliday"].values[hol]) if hol.any() else float("nan")
    nonhol_wmae = float(np.mean(np.abs(va[TARGET].values[~hol] - pred[~hol]))) if (~hol).any() else float("nan")

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model", "TimesFM-2.0-500m")
        mlflow.log_param("mode", "zero_shot")
        mlflow.log_param("context_len", context_len or "full")
        mlflow.log_param("horizon", horizon)
        mlflow.log_param("target_space", "log1p" if log_space else "raw")
        mlflow.log_param("n_series", len(uids))
        mlflow.log_param("forecast_coverage", round(matched / len(va), 4))
        mlflow.log_param("n_fallback", n_fallback)
        mlflow.log_param("overfit_gap", "n/a (zero-shot)")
        mlflow.log_metric("val_wmae", val_wmae)
        mlflow.log_metric("val_mae", val_mae)
        mlflow.log_metric("val_rmse", val_rmse)
        mlflow.log_metric("holiday_wmae", hol_wmae)
        mlflow.log_metric("nonholiday_wmae", nonhol_wmae)

    print(f"[{run_name:22s}] val_wmae={val_wmae:8.2f}  hol={hol_wmae:8.2f}  "
          f"nonhol={nonhol_wmae:8.2f}  cov={matched/len(va):.1%}  fb={n_fallback}")
    return val_wmae


mlflow.set_experiment("TimesFM_Training")
run_timesfm("TimesFM_log1p", context_len=156, log_space=True)

🏃 View run TimesFM_log1p at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/8/runs/34980dfdc0ad44be80db4c90f2eac048
🧪 View experiment at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/8
[TimesFM_log1p         ] val_wmae= 1281.07  hol= 1478.59  nonhol= 1204.98  cov=99.9%  fb=23


np.float64(1281.0677025930183)